# 0.1 Import Libraries

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# 0.2 Load Project Modules

In [ ]:
PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from config.settings import RAW_DIR, PROCESSED_DIR, OUTPUT_DIR
from src.data_loader import build_processed_tables, load_processed_table
from src.universe import define_store_universe, identify_active_stores, add_sales_size_group, create_store_strata, build_universe_summaries

# 1.1 Load Raw Data

In [ ]:
RAW_DIR.exists(), sorted(path.name for path in RAW_DIR.glob("*"))

# 1.2 Build Aggregated Sales Tables

In [ ]:
analysis_start_date = "2017-01-01"
analysis_end_date = "2017-08-15"

tables = build_processed_tables(
    RAW_DIR,
    PROCESSED_DIR,
    start_date=analysis_start_date,
    end_date=analysis_end_date,
    chunksize=300_000,
    csv_only=True,
)
tables.keys()

# 1.3 Join Store Metadata

In [ ]:
store_metadata_sales = tables["store_metadata_sales"] if "store_metadata_sales" in tables else load_processed_table(PROCESSED_DIR, "store_metadata_sales")
store_transactions = tables["store_transaction_counts"] if "store_transaction_counts" in tables else load_processed_table(PROCESSED_DIR, "store_transaction_counts")
store_metadata_sales.head()

# 2.1 Define Universe Scope

In [ ]:
universe = define_store_universe(store_metadata_sales)
universe.shape

# 2.2 Identify Active Stores

In [ ]:
universe = identify_active_stores(universe, store_transactions, min_sales_days=30, min_transaction_days=30)
universe["is_active"].value_counts(dropna=False)

# 2.3 Create Store Sales Size Groups

In [ ]:
universe = add_sales_size_group(universe)
universe["sales_size_group"].value_counts()

# 2.4 Create Store Strata

In [ ]:
universe = create_store_strata(universe)
universe[["store_nbr", "store_stratum"]].head()

# 3.1 Universe Summary by Store Type

In [ ]:
summary_by_type = universe.groupby("type", as_index=False).agg(store_count=("store_nbr", "nunique"), total_sales=("total_sales", "sum"))
summary_by_type

# 3.2 Universe Summary by Cluster

In [ ]:
summary_by_cluster = universe.groupby("cluster", as_index=False).agg(store_count=("store_nbr", "nunique"), total_sales=("total_sales", "sum"))
summary_by_cluster.head()

# 3.3 Universe Summary by Geography

In [ ]:
summary_by_geo = universe.groupby(["state", "city"], as_index=False).agg(store_count=("store_nbr", "nunique"), total_sales=("total_sales", "sum"))
summary_by_geo.head()

# 3.4 Universe Summary by Sales Size Group

In [ ]:
summary_by_size = universe.groupby("sales_size_group", as_index=False).agg(store_count=("store_nbr", "nunique"), total_sales=("total_sales", "sum"))
summary_by_size

# 4.1 Save Universe Outputs

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
universe_summary = build_universe_summaries(universe)
universe_summary.to_csv(OUTPUT_DIR / "universe_summary.csv", index=False)
universe[universe["is_active"]].to_csv(OUTPUT_DIR / "active_store_universe.csv", index=False)
universe_summary.head()